# Predicting Macroeconomics Classification and Analysing Development Indices

**Objective:** Compare three Random Forest configurations to understand how removing direct income-related features affects classification of countries as **High income** or **Not high income**.

The three experiments are:
1. All six original features.
2. Remove `GNI_per_Capita`.
3. Remove both `GDP_pc_2023` and `GNI_per_Capita`.

All experiments use the same dataset, stratified 80:20 train-test split (`random_state=42`), preprocessing pipeline, five-fold cross-validation, and GridSearchCV search space. The held-out test set is used only once per fitted experiment for final evaluation.

## 1. Setup and imports

If needed, install the dependencies in a terminal:

`pip install pandas numpy matplotlib seaborn scikit-learn openpyxl`

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Locate and load the project data

In [ ]:
import pandas as pd
from pathlib import Path

def load_data_directly():
    base = Path.cwd()
    metadata_path = base / "Metadata.csv"
    gdp_path = base / "GDP.csv"
    hdi_path = base / "HDI.xlsx"

    if not metadata_path.exists():
        metadata_path = base / "PMC" / "Datasets" / "Metadata.csv"
    if not gdp_path.exists():
        gdp_path = base / "PMC" / "Datasets" / "GDP.csv"
    if not hdi_path.exists():
        hdi_path = base / "PMC" / "Datasets" / "HDI.xlsx"

    if not (metadata_path.exists() and gdp_path.exists() and hdi_path.exists()):
        raise FileNotFoundError(
            f"Could not find all required datasets ('Metadata.csv', 'GDP.csv', 'HDI.xlsx') in "
            f"{base} or PMC/Datasets/."
        )

    metadata = pd.read_csv(metadata_path)
    print("Metadata dataset loaded successfully.")

    gdp = pd.read_csv(gdp_path, skiprows=4)
    print("GDP dataset loaded successfully.")

    hdi_raw = pd.read_excel(hdi_path, skiprows=4, sheet_name="Table 1. HDI")
    print("HDI dataset loaded successfully.")

    return metadata, gdp, hdi_raw

# Load datasets
metadata, gdp, hdi_raw = load_data_directly()
print("\n--- Dataset Shapes & Names Summary ---")
print(f"- Dataset 'Metadata': Shape = {metadata.shape}")
print(f"- Dataset 'GDP': Shape = {gdp.shape}")
print(f"- Dataset 'HDI': Shape = {hdi_raw.shape}")

## 3. Recreate the original data preparation

The following cleaning and merging steps follow the supplied project notebook:
- World Bank country metadata is joined with 2023 GDP per capita.
- UNDP HDI indicators are loaded from the `Table 1. HDI` sheet.
- Countries are joined by their official country names.
- Numeric indicators are cleaned and converted to numeric values.
- Rows missing any of the six original features or the income group are removed.
- The target is `1` for High income and `0` otherwise.

In [ ]:
metadata_clean = metadata[["Country Code", "IncomeGroup", "TableName"]].dropna(
    subset=["IncomeGroup"]
).copy()
metadata_clean["TableName"] = metadata_clean["TableName"].str.strip()

gdp_clean = gdp[["Country Code", "2023"]].rename(
    columns={"2023": "GDP_pc_2023"}
).dropna().copy()
world_bank = metadata_clean.merge(gdp_clean, on="Country Code", how="inner")

hdi = hdi_raw.iloc[3:].copy()
hdi.columns = [
    "HDI_Rank", "Country", "HDI_Value", "empty1", "Life_Expectancy", "empty2",
    "Expected_Schooling", "empty3", "Mean_Schooling", "empty4",
    "GNI_per_Capita", "empty5", "GNI_minus_HDI_Rank", "empty6", "HDI_Rank_2022"
]
hdi = hdi[[
    "Country", "HDI_Value", "Life_Expectancy", "Expected_Schooling",
    "Mean_Schooling", "GNI_per_Capita"
]].dropna(subset=["Country", "HDI_Value"]).copy()
hdi["Country"] = hdi["Country"].astype(str).str.strip()

merged = world_bank.merge(hdi, left_on="TableName", right_on="Country", how="inner")

FEATURES = [
    "GDP_pc_2023", "HDI_Value", "Life_Expectancy",
    "Expected_Schooling", "Mean_Schooling", "GNI_per_Capita"
]
TARGET = "Target"

for col in FEATURES:
    merged[col] = (
        merged[col].astype(str)
        .str.replace(r"[^²\d\.]", "", regex=True)
    )
    merged[col] = pd.to_numeric(merged[col], errors="coerce")
display(merged.head())
data = merged.dropna(subset=FEATURES + ["IncomeGroup"]).copy()
data[TARGET] = data["IncomeGroup"].astype(str).str.contains(
    "High income", case=False
).astype(int)


print("\nFirst five rows of the final cleaned dataset:")
display(data[["TableName", "IncomeGroup"] + FEATURES + [TARGET]].head(5))

print("\nIncome Category Counts:")
display(data[TARGET].value_counts().rename(index={0: "Not high income", 1: "High income"}))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Output 3: Income Distribution ---
plt.figure(figsize=(8, 5))
ax = sns.countplot(
    x=data[TARGET].map({0: "Not high income", 1: "High income"}),
    order=["Not high income", "High income"],
    palette="Set2"
)

# Add exact counts on top of the bars
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom',
        fontsize=11, fontweight='bold', color='black',
        xytext=(0, 5),
        textcoords='offset points'
    )

plt.title("Distribution of Target Income Categories", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Income Classification Category", fontsize=12)
plt.ylabel("Number of Countries", fontsize=12)
plt.ylim(0, max(data[TARGET].value_counts()) * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Output 4: Correlation Heatmap ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
correlation_matrix = data[FEATURES].corr()
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title("Correlation Heatmap of Original Economic and Development Features", fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Define one shared train-test split

A single stratified split is created before running the experiments. Every configuration therefore uses exactly the same training and test countries. Stratification preserves the class proportions in both subsets.

In [ ]:
X = data[FEATURES].copy()
y = data[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("\\nTraining class proportions:")
display(y_train.value_counts(normalize=True).rename(index={0: "Not high income", 1: "High income"}))
print("Test class proportions:")
display(y_test.value_counts(normalize=True).rename(index={0: "Not high income", 1: "High income"}))

## 5. Configure the three experiments

| Experiment | Features removed | Features retained |
|---|---|---|
| 1 — Original | None | All six |
| 2 — Without GNI | `GNI_per_Capita` | Five features |
| 3 — Without GDP & GNI | `GDP_pc_2023`, `GNI_per_Capita` | Four features |

**Important interpretation note:** HDI is a composite index that includes GNI per capita as one of its dimensions. Therefore, Experiment 3 removes the two standalone GDP/GNI columns, but it does **not** remove all income-related information: some income information remains embedded in HDI. Describe this experiment as using the remaining development indicators plus HDI, rather than as a completely income-free model.

In [ ]:
EXPERIMENTS = {
    "All 6 Features": FEATURES.copy(),
    "Without GNI": [f for f in FEATURES if f != "GNI_per_Capita"],
    "Without GDP & GNI": [
        f for f in FEATURES if f not in ["GDP_pc_2023", "GNI_per_Capita"]
    ],
}

for experiment, selected_features in EXPERIMENTS.items():
    print(f"{experiment}: {selected_features}")

## 6. Preprocessing, cross-validation and hyperparameter tuning

Each experiment uses the same pipeline:
1. Median imputation (kept inside the pipeline to avoid fitting preprocessing on validation/test data).
2. Standard scaling (kept consistent with the supplied project workflow).
3. Random Forest with `class_weight="balanced"` and `random_state=42`.

GridSearchCV uses five-fold stratified cross-validation on the training data only. The best configuration is selected using **F1-score**. The test set is not involved in hyperparameter selection.

In [ ]:
PARAM_GRID = {
    "model__n_estimators": [100, 250],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2],
}

cv_strategy = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)

def build_pipeline(selected_features):
    numeric_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    preprocessor = ColumnTransformer(
        transformers=[("numeric", numeric_preprocessor, selected_features)],
        remainder="drop",
    )
    random_forest = RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", random_forest),
    ])

## 7. Run all three experiments

This cell fits three separate GridSearchCV searches. Each search sees the same training rows and the same folds, but receives only the feature columns belonging to its experiment.

In [ ]:
experiment_results = {}

for experiment_name, selected_features in EXPERIMENTS.items():
    print("\\n" + "=" * 75)
    print(experiment_name)
    print("=" * 75)

    search = GridSearchCV(
        estimator=build_pipeline(selected_features),
        param_grid=PARAM_GRID,
        scoring="f1",
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
        verbose=0,
        return_train_score=True,
    )
    search.fit(X_train[selected_features], y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test[selected_features])
    y_probability = best_model.predict_proba(X_test[selected_features])[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_probability),
    }

    experiment_results[experiment_name] = {
        "features": selected_features,
        "search": search,
        "model": best_model,
        "y_pred": y_pred,
        "y_probability": y_probability,
        "metrics": metrics,
    }

    print(f"Best mean CV F1: {search.best_score_:.4f}")
    print("Best parameters:", search.best_params_)
    print("Held-out test metrics:")
    for metric, value in metrics.items():
        print(f"  {metric:10s}: {value:.4f} ({value:.2%})")

## 8. Compare the three models

The table below reports actual held-out test metrics calculated from the supplied dataset and the experiments above. Values are not estimated or manually filled in.

In [ ]:
# --- Output 5: Model Comparison ---
import pandas as pd

print("=== Output 5: Model Comparison (Held-out Test Performance) ===")
comparison = pd.DataFrame({
    name: result["metrics"]
    for name, result in experiment_results.items()
}).T

comparison_percent = (comparison * 100).round(2)
comparison_percent.index.name = "Experiment Configuration"

display(comparison_percent.style.format("{:.2f}%").set_caption(
    "Held-out Test Performance Comparison Table (%)"
))

print("\n--- Cross-Validation and Hyperparameter Parameters ---")
cv_comparison = pd.DataFrame({
    name: {
        "Best mean 5-fold CV F1": f"{result['search'].best_score_:.4f}",
        "Best parameters": str(result["search"].best_params_),
        "Number of features": len(result["features"]),
    }
    for name, result in experiment_results.items()
}).T
display(cv_comparison)

In [ ]:
from sklearn.metrics import classification_report

# --- Output 6: Best Model Classification Report ---
print("=== Output 6: Best Model Classification Report ('All 6 Features' Configuration) ===\n")
best_result = experiment_results["All 6 Features"]
report_str = classification_report(
    y_test,
    best_result["y_pred"],
    target_names=["Not high income", "High income"],
    zero_division=0
)
print(report_str)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- Output 7: Best Model Confusion Matrix ---
plt.figure(figsize=(6, 5))
best_cm = confusion_matrix(y_test, best_result["y_pred"])

sns.heatmap(
    best_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not high income", "High income"],
    yticklabels=["Not high income", "High income"],
    cbar=False,
    annot_kws={"size": 14, "weight": "bold"}
)

plt.title("Confusion Matrix: Best Model ('All 6 Features')", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Predicted Category", fontsize=11)
plt.ylabel("True Category", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# --- Output 8: ROC Curves Comparison ---
plt.figure(figsize=(6, 5))

for name, result in experiment_results.items():
    fpr, tpr, _ = roc_curve(y_test, result["y_probability"])
    auc_value = result["metrics"]["ROC-AUC"]
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_value:.4f})", linewidth=2)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Baseline (AUC = 0.50)")
plt.title("ROC Curves comparison — Three Feature Configurations", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right", fontsize=10.5)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Output 9: Feature Importance for the Best Model ---
best_model_pipeline = experiment_results["All 6 Features"]["model"]
rf_estimator = best_model_pipeline.named_steps["model"]
feature_cols = experiment_results["All 6 Features"]["features"]

# Calculate and sort importance
importances = pd.Series(rf_estimator.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
sns.barplot(
    x=importances.values,
    y=importances.index,
    palette="viridis",
    edgecolor="black"
)

# Add values on the bars
for index, value in enumerate(importances.values):
    plt.text(value + 0.01, index, f"{value:.4f}", va="center", fontsize=10.5, fontweight="bold")

plt.title("Feature Importance: Best Model ('All 6 Features' Configuration)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Relative Importance Score", fontsize=11)
plt.ylabel("Input Features", fontsize=11)
plt.xlim(0, max(importances.values) * 1.15)
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- Output 10: Final Prediction Details Showcase ---
import pandas as pd

print("=== Output 10: Final Prediction Details Showcase (Subset of Test Set) ===\n")

# Create a showcase dataframe using test indices
showcase_df = data.loc[y_test.index, ["TableName", "IncomeGroup"]].copy()
showcase_df["Actual Class"] = y_test.map({0: "Not high income", 1: "High income"})
showcase_df["Predicted Class"] = pd.Series(best_result["y_pred"], index=y_test.index).map({0: "Not high income", 1: "High income"})
showcase_df["Probability of High Income"] = best_result["y_probability"]

# Formatting output
showcase_df["Probability of High Income"] = showcase_df["Probability of High Income"].apply(lambda x: f"{x:.2%}")
showcase_df.rename(columns={"TableName": "Country/Entity", "IncomeGroup": "World Bank Class"}, inplace=True)

# Display the top 10 rows for high-quality screenshot capture
display(showcase_df.head(10).style.set_caption("Sample Test Set Predictions & Confidence Levels — Best Model"))

## 9. Compare confusion matrices and ROC curves

These plots help explain the metric differences. The confusion matrix shows correct and incorrect classifications; the ROC curve compares true-positive and false-positive rates across probability thresholds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, result) in zip(axes, experiment_results.items()):
    cm = confusion_matrix(y_test, result["y_pred"])
    ConfusionMatrixDisplay(
        cm, display_labels=["Not high income", "High income"]
    ).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(name)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for name, result in experiment_results.items():
    fpr, tpr, _ = roc_curve(y_test, result["y_probability"])
    auc_value = result["metrics"]["ROC-AUC"]
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — three feature configurations")
plt.legend()
plt.show()

## 10. Compare feature importance

Feature importance is calculated from each experiment's tuned Random Forest. Importance values are relative within a model and should not be interpreted as causal effects. Since the feature sets differ, compare patterns cautiously.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, result) in zip(axes, experiment_results.items()):
    estimator = result["model"].named_steps["model"]
    importance = pd.Series(
        estimator.feature_importances_,
        index=result["features"]
    ).sort_values()
    importance.plot(kind="barh", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 11. Compare with the results already reported in the project documentation

The documentation supplied with the project reports the following Random Forest results for the original six-feature model. The experiment above independently re-runs the original configuration using the supplied ZIP data and the stated split/tuning procedure.

In [ ]:
documented_original = pd.Series({
    "Accuracy": 0.9697,
    "Precision": 1.0000,
    "Recall": 0.9167,
    "F1-score": 0.9565,
    "ROC-AUC": 0.9425,
}, name="Documentation")

reproduced_original = pd.Series(
    experiment_results["All 6 Features"]["metrics"],
    name="Reproduced experiment",
)
validation_table = pd.concat([documented_original, reproduced_original], axis=1)
validation_table["Absolute difference (percentage points)"] = (
    validation_table["Reproduced experiment"] - validation_table["Documentation"]
).abs() * 100
display(validation_table.style.format({
    "Documentation": "{:.4f}",
    "Reproduced experiment": "{:.4f}",
    "Absolute difference (percentage points)": "{:.4f}",
}))

## 12. Automatically summarize the experiment results

The summary below identifies the best model for each metric from the held-out test results. This is descriptive reporting only; do not use the test set to retune or choose another model.

In [ ]:
best_by_metric = comparison.idxmax()
print("Best experiment by held-out test metric:")
for metric, experiment in best_by_metric.items():
    print(f"- {metric}: {experiment} ({comparison.loc[experiment, metric]:.4f})")

print("\\nTest-set sample size:", len(y_test))
print("Positive class (High income) in test set:", int(y_test.sum()))
print("Negative class (Not high income) in test set:", int((y_test == 0).sum()))

print("\\nClassification reports:")
for name, result in experiment_results.items():
    print("\\n", name)
    print(classification_report(
        y_test, result["y_pred"],
        target_names=["Not high income", "High income"],
        zero_division=0,
    ))

## 13. Findings and interpretation

Use the actual comparison table above as the source of numerical results. In interpreting it, keep these points in mind:

- **Experiment 1** is the original six-feature baseline.
- **Experiment 2** tests whether the model's performance changes when the standalone GNI per capita feature is removed.
- **Experiment 3** tests the contribution of the standalone GDP and GNI columns together.
- If Experiment 1 and Experiment 2 perform similarly, the standalone GNI feature may add limited incremental value when the other indicators are present. This does not prove GNI is irrelevant.
- A change in ROC-AUC can occur even when threshold-based metrics such as accuracy and F1 remain the same, because ROC-AUC evaluates ranking across thresholds.
- Experiment 3 still contains HDI, which itself includes an income component. It is therefore not a pure test of non-income human-development variables.
- The dataset contains a relatively small number of countries, so a small change in predictions can move the reported metrics noticeably.

### Important methodological limitation

The target is derived from the World Bank `IncomeGroup`, while GDP per capita and GNI per capita are closely related to how income groups are defined. This creates a risk of **target leakage / target-proxy dependence** in the original model. The feature-removal experiments are useful diagnostics, but they do not by themselves establish that the model can predict income classification independently of income information.

### Suggested conclusion for the project report

Report all three test results in one table and explain the observed trade-off. Do not claim that removing a feature improves generalization based on this single test split alone. For a stronger conclusion, repeat the experiment across multiple stratified splits or use an external later-year dataset for validation.

## 14. Results from this run

The experiments were executed on **164 countries**. The stratified 80:20 split produced **131 training rows** and **33 test rows**.

| Metric | All 6 Features | Without GNI | Without GDP & GNI |
|---|---:|---:|---:|
| Accuracy | 96.97% | 96.97% | 90.91% |
| Precision | 100.00% | 100.00% | 90.91% |
| Recall | 91.67% | 91.67% | 83.33% |
| F1-score | 95.65% | 95.65% | 86.96% |
| ROC-AUC | 94.25% | 93.65% | 95.24% |

**Interpretation:** Removing GNI alone did not change the test-set accuracy, precision, recall, or F1-score; ROC-AUC decreased slightly. Removing both GDP and GNI reduced accuracy, recall, and F1-score, while ROC-AUC was higher. This shows that threshold-based classification and ranking performance can move in different directions.

The original six-feature experiment reproduced the metrics recorded in the project documentation on this dataset and split. These results are from one fixed train-test split; they should not be presented as proof of performance on unseen future years or other datasets.